# Validation (b): MZI mesh vs the Clements 2016 decomposition

**Claim checked.** `awen_physics.clements.decompose` implements the rectangular
decomposition of W. R. Clements *et al.*, "Optimal design for universal multiport
interferometers," *Optica* **3**, 1460 (2016), doi:10.1364/OPTICA.3.001460. For
Haar-random unitaries (F. Mezzadri, *Notices of the AMS* **54**, 592 (2007)):

* the mesh uses N(N−1)/2 MZIs with optical depth N (N ≥ 3), as in the paper;
* every unit is unitary;
* the reconstruction error max|U − U_mesh| is below 1×10⁻¹² for N ≤ 32.

**Evidence level.** L0 (math reference): exact linear algebra on ideal MZIs; no
loss, no device data.

**Sources.** Printed errors and fidelities: measured by us (linear-algebra output;
stored with command, commit and environment by
`python scripts/run_validation_notebooks.py`). Mesh structure: peer-reviewed
(Clements *et al.* 2016). Phase-error σ values are a sweep, not device data.

**Fidelity.** Analytic (exact linear algebra). The phase-error study below is a
compact model of programming error, not a measurement.

In [ ]:
import numpy as np
from awen_physics.clements import decompose, haar_unitary, mzi, fidelity

rng = np.random.default_rng(2016)
print(" N  MZIs  depth  max|U-U_mesh|  unitarity error")
for n in (2, 3, 4, 6, 8, 16, 32):
    u = haar_unitary(n, rng)
    mesh = decompose(u)
    rebuilt = mesh.matrix()
    reconstruction = np.abs(rebuilt - u).max()
    unitarity = np.abs(rebuilt.conj().T @ rebuilt - np.eye(n)).max()
    print(f"{n:2d}  {len(mesh.units):4d}  {mesh.depth():5d}  {reconstruction:13.2e}  {unitarity:.2e}")
    assert len(mesh.units) == n * (n - 1) // 2
    assert mesh.depth() == (n if n >= 3 else 1)
    assert reconstruction < 1e-12 and unitarity < 1e-12
    for unit in mesh.units:
        block = mzi(unit.theta, unit.phi)
        assert np.allclose(block.conj().T @ block, np.eye(2), atol=1e-14)

In [ ]:
# Sensitivity to random phase-programming error (compact model): fidelity
# |Tr(U† U_err)|²/N² for an 8-mode mesh, averaged over 50 error draws.
n = 8
u = haar_unitary(n, rng)
mesh = decompose(u)
for sigma in (0.0, 0.001, 0.01, 0.05):
    values = [fidelity(u, mesh.matrix(rng.normal(0, sigma, size=(len(mesh.units), 2)))) for _ in range(50)]
    print(f"σ = {sigma:<6} rad  mean fidelity {np.mean(values):.6f}")
    if sigma == 0.0:
        assert np.mean(values) > 1 - 1e-12
assert np.mean(values) < 1.0  # errors reduce fidelity

**Result.** The decomposition reproduces the paper's structure: N(N−1)/2 units,
depth N, and reconstruction at machine precision. The fidelity falls as phase
error grows. These values come from the model and do not describe any
fabricated mesh.